In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv("../data/SeoulBikeData.csv", encoding="latin1")
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")

# Keep functioning days and sort chronologically
df = df[df["Functioning Day"] == "Yes"].sort_values(["Date", "Hour"])

# Create additional features
df["DayOfWeek"] = df["Date"].dt.day_of_week 
df["IsHoliday"] = (df["Holiday"] == "Holiday").astype(int)

features = [
    "Hour",
    "Temperature(°C)",
    "Rainfall(mm)",
    "Humidity(%)",
    "IsHoliday",
    "DayOfWeek"
]

X = df[features]
y = df["Rented Bike Count"]

# Train / Validation / Test split
train_end = int(len(X) * 0.70)
val_end = int(len(X) * 0.85)

X_train = X.iloc[:train_end]
X_val = X.iloc[train_end:val_end]
X_test = X.iloc[val_end:]

y_train = y.iloc[:train_end]
y_val = y.iloc[train_end:val_end]
y_test = y.iloc[val_end:]

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Train Random Forest
forest = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

forest.fit(X_train, y_train)

# Predict
train_pred = forest.predict(X_train)
val_pred = forest.predict(X_val)

# Evaluate
train_mae = mean_absolute_error(y_train, train_pred)
val_mae = mean_absolute_error(y_val, val_pred)

print(f"Train MAE:         {train_mae:.2f}")
print(f"Validation MAE:    {val_mae:.2f}")
print(f"Decision Tree MAE: 206.63")

In [ ]:
depths = range(4, 21)
train_errors = []
val_errors = []

for depth in depths:
    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=depth,
        random_state=42
    )

    model.fit(X_train, y_train)

    train_errors.append(
        mean_absolute_error(y_train, model.predict(X_train))
    )
    val_errors.append(
        mean_absolute_error(y_val, model.predict(X_val))
    )

plt.figure(figsize=(12, 5))
plt.plot(depths, train_errors, marker="o", label="Train MAE")
plt.plot(depths, val_errors, marker="o", label="Validation MAE")

plt.title("Random Forest: MAE vs Max Depth")
plt.xlabel("Max Depth")
plt.ylabel("MAE")
plt.xticks(depths)
plt.legend()
plt.grid(True)
plt.show()

best_depth = depths[val_errors.index(min(val_errors))]

print(f"Best depth: {best_depth}")
print(f"Best Validation MAE: {min(val_errors):.2f}")

The best max_depth is 14, but the improvement is very small. After depth 14, the validation error stays almost the same, while the training error keeps decreasing.

In [ ]:
tree_counts = [10, 20, 50, 100, 150, 200, 300]

train_errors = []
val_errors = []

for count in tree_counts:
    model = RandomForestRegressor(
        n_estimators=count,
        max_depth=14,
        random_state=42
    )

    model.fit(X_train, y_train)

    train_errors.append(
        mean_absolute_error(y_train, model.predict(X_train))
    )
    val_errors.append(
        mean_absolute_error(y_val, model.predict(X_val))
    )

plt.figure(figsize=(12, 5))
plt.plot(tree_counts, train_errors, marker="o", label="Train MAE")
plt.plot(tree_counts, val_errors, marker="o", label="Validation MAE")

plt.title("Random Forest: MAE vs Number of Trees")
plt.xlabel("Number of Trees")
plt.ylabel("MAE")
plt.xticks(tree_counts)
plt.legend()
plt.grid(True)
plt.show()

best_count = tree_counts[val_errors.index(min(val_errors))]

print(f"Best n_estimators: {best_count}")
print(f"Best Validation MAE: {min(val_errors):.2f}")

After about 50 trees, adding more trees barely improves the result. So 50 seems like a good choice instead of using 300.

In [ ]:
# Weather features + Hour
weather_features = [
    "Temperature(°C)",
    "Humidity(%)",
    "Wind speed (m/s)",
    "Visibility (10m)",
    "Dew point temperature(°C)",
    "Solar Radiation (MJ/m2)",
    "Rainfall(mm)",
    "Snowfall (cm)"
]

features_weather = ["Hour"] + weather_features

X_weather = df[features_weather]

weather_forest = RandomForestRegressor(
    n_estimators=50,
    max_depth=14,
    random_state=42
)

weather_forest.fit(X_weather.iloc[:train_end], y_train)

weather_pred = weather_forest.predict(
    X_weather.iloc[train_end:val_end]
)

weather_mae = mean_absolute_error(y_val, weather_pred)

print(f"Weather + Hour Validation MAE: {weather_mae:.2f}")

In [ ]:
features_extended = features_weather + ["DayOfWeek", "IsHoliday"]

X_extended = df[features_extended]

extended_forest = RandomForestRegressor(
    n_estimators=50,
    max_depth=14,
    random_state=42
)

extended_forest.fit(X_extended.iloc[:train_end], y_train)

extended_pred = extended_forest.predict(
    X_extended.iloc[train_end:val_end]
)

extended_mae = mean_absolute_error(y_val, extended_pred)

print(f"Weather + Hour MAE: {weather_mae:.2f}")
print(f"With DayOfWeek + IsHoliday MAE: {extended_mae:.2f}")

In [ ]:
from sklearn.model_selection import train_test_split

x_compare = df[weather_features + ["Hour"]]

x_train_random, x_test_random, y_train_random, y_test_random = train_test_split(
    x_compare,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(x_train_random, y_train_random)

predictions = model.predict(x_test_random)
mae_random = mean_absolute_error(y_test_random, predictions)

print(f"Random Split MAE: {mae_random:.2f}")